In [1]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
import lightgbm as lgbm
import optuna
import pandas as pd
import joblib

In [2]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_val = pd.read_parquet('../data/X_val.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet').squeeze()
y_val = pd.read_parquet('../data/y_val.parquet').squeeze()
y_test = pd.read_parquet('../data/y_test.parquet').squeeze()

In [11]:
training_data = lgbm.Dataset(X_train, label=y_train, free_raw_data=False)
validation_data = lgbm.Dataset(X_val, label=y_val, free_raw_data=False)

### 1. Hyperparameter Tuning

In [12]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",

        "verbose": -1,
        "is_unbalance": True,
        "feature_pre_filter": False,
                
        "n_jobs": -1,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.09),
        "num_leaves": trial.suggest_int("num_leaves", 150, 300),
        "max_depth": trial.suggest_int("max_depth", 10, 14),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 150),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 1.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 1.0)
    }
    
    callbacks = [lgbm.early_stopping(stopping_rounds=50), lgbm.log_evaluation(100)]
    
    model = lgbm.train(
        params, 
        training_data, 
        valid_sets=[validation_data],
        num_boost_round=3000,
        callbacks=callbacks
    )
    
    return model.best_score["valid_0"]["auc"]

In [13]:
study = optuna.create_study(direction="maximize")

[I 2026-06-26 17:19:38,336] A new study created in memory with name: no-name-76667c9d-4518-4cb3-b7c3-d2eb7c0f7e57


In [15]:
study.optimize(objective, n_trials=10)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.708714
[200]	valid_0's auc: 0.713646
[300]	valid_0's auc: 0.716297
[400]	valid_0's auc: 0.717916
[500]	valid_0's auc: 0.71921
[600]	valid_0's auc: 0.720296
[700]	valid_0's auc: 0.721178
[800]	valid_0's auc: 0.722187
[900]	valid_0's auc: 0.722923
[1000]	valid_0's auc: 0.72358
[1100]	valid_0's auc: 0.72415
[1200]	valid_0's auc: 0.724632
[1300]	valid_0's auc: 0.724965
[1400]	valid_0's auc: 0.725485
[1500]	valid_0's auc: 0.72592
[1600]	valid_0's auc: 0.726291
[1700]	valid_0's auc: 0.726742
[1800]	valid_0's auc: 0.727088
[1900]	valid_0's auc: 0.727277
[2000]	valid_0's auc: 0.727479
[2100]	valid_0's auc: 0.727681
[2200]	valid_0's auc: 0.727886
[2300]	valid_0's auc: 0.728083
[2400]	valid_0's auc: 0.728219
[2500]	valid_0's auc: 0.728369
[2600]	valid_0's auc: 0.728571
[2700]	valid_0's auc: 0.728759
[2800]	valid_0's auc: 0.728892
[2900]	valid_0's auc: 0.729028
[3000]	valid_0's auc: 0.729163
Did not meet early sto

[I 2026-06-26 17:26:40,469] Trial 1 finished with value: 0.7291639099518086 and parameters: {'learning_rate': 0.07194074158181878, 'num_leaves': 187, 'max_depth': 10, 'min_child_samples': 125, 'lambda_l1': 0.6060996876635133, 'lambda_l2': 0.1770679784741126}. Best is trial 1 with value: 0.7291639099518086.


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.711167
[200]	valid_0's auc: 0.716591
[300]	valid_0's auc: 0.718852
[400]	valid_0's auc: 0.720304
[500]	valid_0's auc: 0.72171
[600]	valid_0's auc: 0.722622
[700]	valid_0's auc: 0.723656
[800]	valid_0's auc: 0.724402
[900]	valid_0's auc: 0.725056
[1000]	valid_0's auc: 0.725583
[1100]	valid_0's auc: 0.726135
[1200]	valid_0's auc: 0.726596
[1300]	valid_0's auc: 0.727015
[1400]	valid_0's auc: 0.727361
[1500]	valid_0's auc: 0.727708
[1600]	valid_0's auc: 0.727965
[1700]	valid_0's auc: 0.728127
[1800]	valid_0's auc: 0.728464
[1900]	valid_0's auc: 0.728719
[2000]	valid_0's auc: 0.728974
[2100]	valid_0's auc: 0.729074
[2200]	valid_0's auc: 0.729188
[2300]	valid_0's auc: 0.729274
[2400]	valid_0's auc: 0.72945
[2500]	valid_0's auc: 0.729583
[2600]	valid_0's auc: 0.729614
[2700]	valid_0's auc: 0.729694
[2800]	valid_0's auc: 0.729787
[2900]	valid_0's auc: 0.729832
[3000]	valid_0's auc: 0.729936
Did not meet early s

[I 2026-06-26 17:32:38,612] Trial 2 finished with value: 0.7299386990852624 and parameters: {'learning_rate': 0.07500035010825759, 'num_leaves': 201, 'max_depth': 14, 'min_child_samples': 99, 'lambda_l1': 0.5467795485716909, 'lambda_l2': 0.4150733924827259}. Best is trial 2 with value: 0.7299386990852624.


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.709847
[200]	valid_0's auc: 0.71489
[300]	valid_0's auc: 0.717471
[400]	valid_0's auc: 0.719156
[500]	valid_0's auc: 0.720405
[600]	valid_0's auc: 0.721692
[700]	valid_0's auc: 0.722563
[800]	valid_0's auc: 0.723373
[900]	valid_0's auc: 0.72415
[1000]	valid_0's auc: 0.72482
[1100]	valid_0's auc: 0.725263
[1200]	valid_0's auc: 0.72578
[1300]	valid_0's auc: 0.726251
[1400]	valid_0's auc: 0.726581
[1500]	valid_0's auc: 0.726946
[1600]	valid_0's auc: 0.727249
[1700]	valid_0's auc: 0.727482
[1800]	valid_0's auc: 0.727766
[1900]	valid_0's auc: 0.727868
[2000]	valid_0's auc: 0.728158
[2100]	valid_0's auc: 0.728372
[2200]	valid_0's auc: 0.72843
[2300]	valid_0's auc: 0.728483
[2400]	valid_0's auc: 0.728714
[2500]	valid_0's auc: 0.728817
[2600]	valid_0's auc: 0.728933
[2700]	valid_0's auc: 0.728972
[2800]	valid_0's auc: 0.729002
Early stopping, best iteration is:
[2838]	valid_0's auc: 0.729021


...

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.711058
[200]	valid_0's auc: 0.716109
[300]	valid_0's auc: 0.718487
[400]	valid_0's auc: 0.719986
[500]	valid_0's auc: 0.72114
[600]	valid_0's auc: 0.722159
[700]	valid_0's auc: 0.723157
[800]	valid_0's auc: 0.72389
[900]	valid_0's auc: 0.724857
[1000]	valid_0's auc: 0.725409
[1100]	valid_0's auc: 0.725822
[1200]	valid_0's auc: 0.726153
[1300]	valid_0's auc: 0.726624
[1400]	valid_0's auc: 0.726994
[1500]	valid_0's auc: 0.727259
[1600]	valid_0's auc: 0.727598
[1700]	valid_0's auc: 0.727866
[1800]	valid_0's auc: 0.72816
[1900]	valid_0's auc: 0.728438
[2000]	valid_0's auc: 0.728644
[2100]	valid_0's auc: 0.728829
[2200]	valid_0's auc: 0.729035
[2300]	valid_0's auc: 0.729275
[2400]	valid_0's auc: 0.729366
[2500]	valid_0's auc: 0.729525
[2600]	valid_0's auc: 0.729616
[2700]	valid_0's auc: 0.729756
[2800]	valid_0's auc: 0.729776
Early stopping, best iteration is:
[2758]	valid_0's auc: 0.729778


[I 2026-06-26 18:15:07,107] Trial 9 finished with value: 0.7297783687058609 and parameters: {'learning_rate': 0.06889259251368572, 'num_leaves': 216, 'max_depth': 13, 'min_child_samples': 56, 'lambda_l1': 0.18928720791933684, 'lambda_l2': 0.10764988134042874}. Best is trial 4 with value: 0.7300887353489859.


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.709267
[200]	valid_0's auc: 0.713863
[300]	valid_0's auc: 0.716471
[400]	valid_0's auc: 0.718092
[500]	valid_0's auc: 0.719378
[600]	valid_0's auc: 0.720566
[700]	valid_0's auc: 0.72197
[800]	valid_0's auc: 0.722805
[900]	valid_0's auc: 0.723477
[1000]	valid_0's auc: 0.7242
[1100]	valid_0's auc: 0.724618
[1200]	valid_0's auc: 0.725124
[1300]	valid_0's auc: 0.725491
[1400]	valid_0's auc: 0.725872
[1500]	valid_0's auc: 0.726207
[1600]	valid_0's auc: 0.726616
[1700]	valid_0's auc: 0.726977
[1800]	valid_0's auc: 0.727227
[1900]	valid_0's auc: 0.727665
[2000]	valid_0's auc: 0.727969
[2100]	valid_0's auc: 0.728197
[2200]	valid_0's auc: 0.72848
[2300]	valid_0's auc: 0.728668
[2400]	valid_0's auc: 0.728869
[2500]	valid_0's auc: 0.729022
[2600]	valid_0's auc: 0.729166
[2700]	valid_0's auc: 0.729281
[2800]	valid_0's auc: 0.729413
[2900]	valid_0's auc: 0.729514


[I 2026-06-26 18:20:43,300] Trial 10 finished with value: 0.7296275024686575 and parameters: {'learning_rate': 0.07388645006353645, 'num_leaves': 160, 'max_depth': 12, 'min_child_samples': 141, 'lambda_l1': 0.44730360915259004, 'lambda_l2': 0.8867741281678991}. Best is trial 4 with value: 0.7300887353489859.


[3000]	valid_0's auc: 0.729625
Did not meet early stopping. Best iteration is:
[2992]	valid_0's auc: 0.729628


In [17]:
best_params = study.best_params
print("Best Params:", best_params)

Best Params: {'learning_rate': 0.052136197212208434, 'num_leaves': 269, 'max_depth': 12, 'min_child_samples': 103, 'lambda_l1': 0.5005616352490897, 'lambda_l2': 0.33272215183529574}


In [16]:
best_value = study.best_value
print("Best Value:", best_value)

Best Value: 0.7300887353489859


In [18]:
best_model = lgbm.LGBMClassifier(**best_params, objective="binary", metric="auc", boosting_type="gbdt", verbose=-1, n_jobs=-1, is_unbalanced=True, feature_pre_filter=False)        

In [20]:
X_final = pd.concat([X_train, X_val])
y_final = pd.concat([y_train, y_val])

In [21]:
best_model.fit(
    X_final, y_final,
    callbacks=[lgbm.log_evaluation(100)]
)

,boosting_type,'gbdt'
,num_leaves,269
,max_depth,12
,learning_rate,0.052136197212208434
,n_estimators,100
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,103


In [22]:
joblib.dump(best_model, '../models/best_model.pkl')

['../models/best_model.pkl']